# 6 — Non-numeric convergence

**The concept:** convergence does not require floats. Anything supporting `==` converges on
**structural equality** — unchanged since the last sweep means at rest.

This is what lets a decision, a set, or a configuration sit inside a feedback loop alongside the
numbers. It is also the thing most MDAO frameworks cannot express at all.

In [1]:
from smartmdao import Pipeline, IterativeSolver

DEPENDS_ON = {"billing": {"auth", "db"}, "auth": {"db"}, "reporting": {"db"}}

features = Pipeline(solver=IterativeSolver(max_iterations=10, target_var="enabled"))

@features.step(outputs=["enabled"])
def resolve(requested: frozenset, enabled: frozenset) -> frozenset:
    expanded = set(enabled) | set(requested)
    for feature in list(expanded):
        expanded |= DEPENDS_ON.get(feature, set())
    return frozenset(expanded)

out = features.run(requested=frozenset({"billing"}), enabled=frozenset())
print("enabled:", sorted(out["enabled"]))
print("settled in", out["convergence_reports"][-1].iterations, "iterations")

enabled: ['auth', 'billing', 'db']
settled in 2 iterations


The fixed point is reached when a sweep adds nothing new. No tolerance, no residual norm — just
"the set did not change".

## How the residual is computed

For values that do not subtract, the distance is **0.0 if equal, 1.0 if not**. That is all a
convergence check needs: at rest, or not.

In [2]:
from smartmdao import StandardConvergenceChecker

checker = StandardConvergenceChecker()
pairs = [
    (frozenset({"a"}), frozenset({"a"})),
    (frozenset({"a"}), frozenset({"a", "b"})),
    ("cfrp", "cfrp"),
    ("cfrp", "aluminium"),
    (3.0, 3.25),
]
def shown(value):
    """A frozenset prints in hash order, which changes between runs; sort it for display."""
    return f"frozenset({sorted(value)})" if isinstance(value, frozenset) else str(value)

for before, after in pairs:
    print(f"{shown(before):28} -> {shown(after):28} distance = {checker.distance(before, after)}")

frozenset(['a'])             -> frozenset(['a'])             distance = 0.0
frozenset(['a'])             -> frozenset(['a', 'b'])        distance = inf
cfrp                         -> cfrp                         distance = 0.0
cfrp                         -> aluminium                    distance = inf
3.0                          -> 3.25                         distance = 0.25


## A dataclass as the coupling variable

A structured decision converges the same way, as long as it compares by value. A frozen dataclass
does exactly that.

In [3]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Architecture:
    spar: str
    ribs: int

sizing = Pipeline(solver=IterativeSolver(max_iterations=20, target_var="design"))

@sizing.step(outputs=["design"])
def choose(mass_kg: float) -> Architecture:
    if mass_kg > 900:
        return Architecture(spar="cfrp", ribs=8)
    return Architecture(spar="aluminium", ribs=5)

@sizing.step(outputs=["mass_kg"])
def weigh(design: Architecture, payload_kg: float) -> float:
    base = 700.0 if design.spar == "aluminium" else 640.0
    return base + 12.0 * design.ribs + payload_kg

settled = sizing.run(payload_kg=150.0, mass_kg=0.0)
print("design:  ", settled["design"])
print("mass_kg: ", settled["mass_kg"])
print("settled in", settled["convergence_reports"][-1].iterations, "iterations")

Reached max_iterations (20) without converging. Last residual: inf


design:   Architecture(spar='cfrp', ribs=8)
mass_kg:  886.0
settled in 20 iterations


## Mixing numbers and decisions in one loop

Nothing special is required. The numeric part converges on tolerance, the symbolic part on
equality, and the block is done when both are at rest.

In [4]:
mixed = Pipeline(solver=IterativeSolver(max_iterations=40, tolerance=1e-9))

@mixed.step(outputs=["material"])
def pick_material(stress_mpa: float) -> str:
    return "steel" if stress_mpa > 250 else "aluminium"

@mixed.step(outputs=["stress_mpa"])
def stress(material: str, load_kn: float) -> float:
    allowable = 400.0 if material == "steel" else 180.0
    return min(load_kn * 3.0, allowable)

# IterativeSolver sweeps in REGISTRATION order, so `pick_material` runs first
# and reads `stress_mpa` before anything produces it. That is the variable to seed.
mixed_out = mixed.run(load_kn=100.0, stress_mpa=0.0)
print("material:  ", mixed_out["material"])
print("stress_mpa:", mixed_out["stress_mpa"])
print("status:    ", mixed_out["convergence_reports"][-1].status)

material:   aluminium
stress_mpa: 180.0
status:     converged


## The rule that makes this work in practice

**Couple on the decision, never on the explanation of it.**

A coupling variable that carries prose — a rationale string, a timestamp, a log line — will differ
on every sweep even when the decision is identical, and the loop will never settle. Keep the
coupling variable low-entropy: a set, an enum, a frozen dataclass.

In [5]:
noisy = Pipeline(solver=IterativeSolver(max_iterations=5, target_var="verdict"))

counter = {"n": 0}

@noisy.step(outputs=["verdict"])
def decide_with_prose(load_kn: float) -> str:
    counter["n"] += 1
    # The DECISION is stable; the explanation is not.
    return f"steel (evaluated on sweep {counter['n']})"

noisy_report = noisy.run(load_kn=100.0, verdict="")["convergence_reports"][-1]
print("status:    ", noisy_report.status)
print("iterations:", noisy_report.iterations)
print()
print("The decision never changed. The string did, so it never converged.")

Reached max_iterations (5) without converging. Last residual: inf


status:     max_iterations
iterations: 5

The decision never changed. The string did, so it never converged.


---

**Next:** [7 — Type checking](07-type-checking.ipynb).